# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [1]:
import os

from IPython.display import IFrame

url = os.environ.get("DLAI_LOCAL_URL").format(port=8888)
# Terminal 1: uv run a2a_policy_agent.py
IFrame(f"{url}terminals/1", width=800, height=200)

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [2]:
# Terminal 2: uv run a2a_research_agent.py
IFrame(f"{url}terminals/2", width=800, height=200)

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [3]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [5]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [6]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [7]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [8]:
prompt = "How can I claim for visits to GP when I have common health issues like cold cough etc.?"

**Note:** It takes a few seconds for the output to display.

In [9]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

Running Healthcare Workflow Agent


The process for claiming a GP visit for minor issues like a cold or cough depends entirely on your location and the type of healthcare coverage you have.

Because you did not specify your country, I have broken this down by the most common systems.

### **1. United Kingdom (NHS & Private)**
*   **NHS (Public):** You generally **do not claim** for GP visits. They are free at the point of use for residents.
    *   *Alternative:* For colds and coughs, use the **Pharmacy First** scheme. You can go directly to a pharmacist who can give advice and, if necessary, prescription medication for 7 common conditions (including sore throats and earaches) without you needing to see a GP at all.
*   **Health Cash Plans (e.g., Simplyhealth, Westfield):** If you pay for a private "cash plan," you **can** claim.
    1.  Pay for the private GP appointment or prescription upfront.
    2.  Get an itemized receipt (a credit card slip is often not enough).
    3.  Upload a photo of the receipt to your provider’s app or website to get reimbursed.
*   **Private Medical Insurance (PMI):** Most standard PMI policies (like Bupa or AXA) **do not** cover routine GP visits for minor ailments. They are designed for acute conditions requiring specialists/surgery. Check your specific policy terms.

### **2. Australia (Medicare & Private)**
*   **Bulk Billing:** If your doctor "bulk bills," they bill the government directly. You pay nothing and do not need to claim.
*   **Patient Claims (The "Gap"):** If your doctor charges a private fee:
    1.  **Pay the full fee** at the clinic.
    2.  **Claim the rebate:** Most clinics can process this instantly via the EFTPOS terminal. If not, use the **Medicare Express Plus app** or your myGov account to upload the receipt. You will receive a partial refund (the "rebate") into your bank account.
*   **Private Health Insurance:** In Australia, private insurers generally **cannot** cover GP visits that are covered by Medicare. You cannot claim the "gap" fee from your private insurer.

### **3. United States (Private Insurance)**
*   **In-Network Doctors:** You usually do not "claim" yourself. You pay a **co-pay** (e.g., $20) at the front desk, and the doctor’s office files the claim with your insurance company.
*   **Out-of-Network:** If you see a doctor not in your plan:
    1.  Pay the full bill upfront.
    2.  Ask for a "Superbill" (an itemized receipt with medical codes).
    3.  Submit a claim form (available on your insurer’s portal) along with the Superbill. *Note: Reimbursement depends on your out-of-network deductible.*
*   **HSA/FSA:** You can use your Health Savings Account (HSA) or Flexible Spending Account (FSA) card to pay for the visit and over-the-counter medicines (like cough syrup) tax-free.

### **4. Ireland**
*   **Medical Card / GP Visit Card:** The visit is free; you do not need to claim.
*   **Private Patients:** You pay the full fee (typically €45–€65).
    *   **Private Insurance:** Submit a photo of your receipt to your insurer (e.g., Vhi, Laya, Irish Life) via their app. Coverage varies by plan.
    *   **Tax Relief:** If you don't have insurance (or have a shortfall), keep your receipts. You can claim **20% tax relief** on medical expenses by filing a return with Revenue.ie at the end of the year.

### **5. Singapore (Company & Private)**
*   **Company Insurance:** Many employers provide a medical card (e.g., AIA, Fullerton). Show this at a "panel clinic" to pay $0 or a small co-pay. If you visit a non-panel clinic, pay upfront, keep the receipt, and submit a claim via your company’s HR portal.
*   **Integrated Shield Plans:** These generally cover hospitalization, not GP visits for coughs/colds. You typically cannot claim these costs unless you have a specific "rider" or add-on for outpatient care.

### **6. India (Health Insurance)**
*   **OPD Cover:** Standard health insurance usually requires 24-hour hospitalization to trigger a claim. However, if your policy has **OPD (Outpatient Department) cover**, you can claim for GP visits and pharmacy bills.
    *   **Process:** Pay upfront, collect the **bill with the doctor's registration number** and the prescription. Submit these via your insurer's TPA (Third Party Administrator) app for reimbursement.

---

### **Universal "How to Claim" Steps**
If you are in a system where you pay and claim back (Reimbursement Model), follow these steps to ensure your claim isn't rejected:
1.  **Get a Proper Receipt:** A credit card slip is rarely accepted. You need an **invoice/receipt** that shows:
    *   The doctor’s name and license number.
    *   The date of service.
    *   The diagnosis or service code (e.g., "Consultation").
    *   The amount paid.
2.  **Check for "Referrals":** Some policies (especially in Singapore or parts of Europe) require a referral letter to claim for specialists, though usually not for GPs.
3.  **Use the App:** almost all modern insurers (Bupa, Allianz, Manulife, Medicare, etc.) have apps. Taking a photo of the receipt is faster than mailing forms.
4.  **Prescriptions:** If you are claiming for the *medicine* (cough syrup, antibiotics), you usually need the **pharmacy receipt** (listing the drug name) AND a copy of the **doctor's prescription**.

Based on the insurance document provided (Anthem Blue Cross gHIP HDHP plan), here's how you can claim for GP visits for common health issues like colds and coughs:

## Claiming for Primary Care Visits

**For In-Network Providers:**
- You pay **10% coinsurance** after your deductible is met
- Once you meet your deductible (\\$1,700 for individual or \\$3,400 for family), the plan begins to share costs with you

**For Out-of-Network Providers:**
- You pay **30% coinsurance** after your deductible is met
- Your deductible is higher: \\$3,400 for individual or \\$6,800 for family

## Important Process Steps:

1. **See an In-Network Provider** (to minimize costs):
   - Use the networks available: NY Blue Access PPO (if in NY), GA Blue Open Access POS (if in GA), or Blue Card PPO (all other areas)
   - Check provider lists at **includedhealth.com/google** or call **(855) 431-5540**

2. **No Referral Required:**
   - You can visit your primary care doctor directly without needing a referral

3. **Payment at Visit:**
   - If you haven't met your deductible, you typically pay the full cost until the deductible is reached
   - After the deductible is met, you only pay your coinsurance percentage (10% in-network)

4. **No Copay for Preventive Care:**
   - If the visit is determined to be **preventive care/screening**, there is **no charge** even before you meet your deductible

The plan will handle the claim submission directly with in-network providers, so you shouldn't need to submit claims yourself.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>
